# On-Policy Distillation in Colab

This notebook is a cleaned, self-contained Colab workflow for the CS234 on-policy diffusion distillation experiment. It keeps only the pieces needed to:

- generate or load prompts,
- distill `stable-diffusion-3-medium-diffusers` from `stable-diffusion-3.5-medium`,
- save checkpoints and metrics,
- render teacher, original student, and distilled student evaluation images.

The default settings are intentionally small enough for iteration in Colab. Increase the training and image settings only after the smoke test runs cleanly.

In [ ]:
%pip install -q diffusers transformers accelerate sentencepiece safetensors huggingface_hub

import gc
import hashlib
import json
import random
import re
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch
import torch.nn.functional as F
from torch import nn
from torch.amp import GradScaler, autocast

from diffusers import StableDiffusion3Pipeline
from huggingface_hub import notebook_login


In [ ]:
# Optional: uncomment this if the models are gated for your Hugging Face account.
# notebook_login()

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a CUDA runtime in Google Colab.")

EXPERIMENT_NAME = "opd-cs234"
RUN_NAME = "opd-distill-sd35m-sd3m-colab-clean"
SEED = 42

TEACHER_MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
STUDENT_MODEL_ID = "stabilityai/stable-diffusion-3-medium-diffusers"
DTYPE_NAME = "bfloat16"

TRAIN_NUM_INFERENCE_STEPS = 4
EVAL_NUM_INFERENCE_STEPS = 12
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 256
MAX_SEQUENCE_LENGTH = 64
INITIAL_NOISE_SCALE = 1.0

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 1
MAX_TRAIN_STEPS = 40
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
MIXED_PRECISION = True
MAX_GRAD_NORM = 1.0
LOG_EVERY = 1
SAVE_EVERY = 10



FREEZE_TEXT_ENCODERS = True
FREEZE_VAE = True
LOCAL_FILES_ONLY = False
LOW_CPU_MEM_USAGE = True
USE_SAFETENSORS = True
REVISION = None
VARIANT = None
CACHE_DIR = None

GENERATE_PROMPTS = True
NUM_PROMPTS_PER_CATEGORY = 1
PROMPT_CATEGORIES = ["tech"]

EVAL_PROMPTS = [
    "A cozy coffee shop interior at golden hour, cinematic photo style",
    "A futuristic electric sports car driving through a neon city at night",
    "A vibrant fruit tart on a marble table, studio food photography",
    "A hiker standing on a snowy mountain ridge under dramatic clouds",
    "A minimal product ad for wireless earbuds with a clean studio backdrop",
]

OUTPUT_ROOT = Path("/content/opd_outputs")
PROMPTS_JSONL = OUTPUT_ROOT / "prompts" / "prompts.jsonl"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints" / RUN_NAME
METRICS_PATH = OUTPUT_ROOT / "metrics" / RUN_NAME / "metrics.jsonl"
SUMMARY_PATH = OUTPUT_ROOT / "runs" / RUN_NAME / "train_summary.json"
EVAL_OUTPUT_DIR = OUTPUT_ROOT / "eval_images" / RUN_NAME
ZIP_BASENAME = OUTPUT_ROOT / "eval_images"

CATEGORY_TEMPLATES = {
    "fashion": [
        "A premium {adjective} fashion ad for {audience}, featuring {style} styling and studio lighting",
        "A social media banner advertising {adjective} apparel for {audience} with {style} aesthetics",
    ],
    "food": [
        "A mouthwatering ad for {adjective} {product} aimed at {audience}, cinematic close-up",
        "A digital campaign visual promoting {product} with {style} composition and bold typography",
    ],
    "travel": [
        "An aspirational ad for {adjective} travel packages to {destination}, {style} visual style",
        "A conversion-focused ad creative for {destination} vacations targeting {audience}",
    ],
    "tech": [
        "A modern ad for a {adjective} {product} targeting {audience}, rendered in {style} style",
        "A launch campaign visual for {product} with {style} gradients and product hero shot",
    ],
    "fitness": [
        "A high-energy ad promoting {adjective} fitness programs for {audience}, {style} visual language",
        "A performance marketing image for {product} with dynamic motion and {style} palette",
    ],
}
ADJECTIVES = ["minimal", "bold", "luxury", "vibrant", "futuristic", "clean", "playful"]
AUDIENCES = ["young professionals", "college students", "new parents", "small business owners"]
PRODUCTS = ["smartwatch", "meal kit", "running shoes", "wireless earbuds", "protein shake"]
STYLES = ["high-contrast", "editorial", "retro-modern", "photoreal", "flat-design"]
DESTINATIONS = ["Kyoto", "Barcelona", "Iceland", "Bali", "New York City"]

DEVICE = torch.device("cuda")

print(json.dumps({
    "device": str(DEVICE),
    "teacher": TEACHER_MODEL_ID,
    "student": STUDENT_MODEL_ID,
    "output_root": str(OUTPUT_ROOT),
}, indent=2))


In [ ]:
@dataclass
class PromptRecord:
    prompt_id: str
    prompt: str
    category: str
    seed: int


def _resolve_dtype(dtype_name: str) -> torch.dtype:
    mapping = {
        "float16": torch.float16,
        "fp16": torch.float16,
        "float32": torch.float32,
        "fp32": torch.float32,
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
    }
    key = dtype_name.lower()
    if key not in mapping:
        raise ValueError(f"Unsupported dtype: {dtype_name}")
    return mapping[key]


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _prompt_id(category: str, prompt: str, seed: int) -> str:
    digest = hashlib.sha1(f"{category}|{prompt}|{seed}".encode("utf-8")).hexdigest()
    return digest[:16]


def generate_prompt_records(num_per_category: int, seed: int, categories: list[str]) -> list[dict[str, object]]:
    rng = random.Random(seed)
    records = []
    for category in categories:
        templates = CATEGORY_TEMPLATES.get(category)
        if not templates:
            raise ValueError(f"Unknown category: {category}")
        for index in range(num_per_category):
            prompt = rng.choice(templates).format(
                adjective=rng.choice(ADJECTIVES),
                audience=rng.choice(AUDIENCES),
                product=rng.choice(PRODUCTS),
                style=rng.choice(STYLES),
                destination=rng.choice(DESTINATIONS),
            )
            prompt_seed = seed + index
            records.append({
                "prompt_id": _prompt_id(category, prompt, prompt_seed),
                "prompt": prompt,
                "category": category,
                "seed": prompt_seed,
            })
    return records


def save_prompts_jsonl(path: Path, records: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, sort_keys=True) + "\n")


def load_prompts_jsonl(path: Path) -> list[PromptRecord]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            records.append(PromptRecord(
                prompt_id=row["prompt_id"],
                prompt=row["prompt"],
                category=row.get("category", "unknown"),
                seed=int(row.get("seed", SEED)),
            ))
    return records


def iter_prompt_batches(dataset: list[PromptRecord], batch_size: int, seed: int) -> list[list[PromptRecord]]:
    indices = list(range(len(dataset)))
    rng = random.Random(seed)
    rng.shuffle(indices)
    return [
        [dataset[idx] for idx in indices[start : start + batch_size]]
        for start in range(0, len(indices), batch_size)
    ]


def append_metric(path: Path, payload: dict[str, float]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, sort_keys=True) + "\n")


def _slugify(text: str) -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "_", text.strip().lower()).strip("_")
    return cleaned[:60] or "prompt"


def _config_hash() -> str:
    payload = {
        "teacher_model_id": TEACHER_MODEL_ID,
        "student_model_id": STUDENT_MODEL_ID,
        "dtype": DTYPE_NAME,
        "train_num_inference_steps": TRAIN_NUM_INFERENCE_STEPS,
        "max_train_steps": MAX_TRAIN_STEPS,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()


def clear_cuda() -> None:
    gc.collect()
    torch.cuda.empty_cache()


if GENERATE_PROMPTS:
    records = generate_prompt_records(NUM_PROMPTS_PER_CATEGORY, SEED, PROMPT_CATEGORIES)
    save_prompts_jsonl(PROMPTS_JSONL, records)
    print(f"Wrote {len(records)} prompts to {PROMPTS_JSONL}")

dataset = load_prompts_jsonl(PROMPTS_JSONL)
if not dataset:
    raise RuntimeError(f"No prompts found at {PROMPTS_JSONL}")

batches = iter_prompt_batches(dataset, BATCH_SIZE, SEED)
print(f"Loaded {len(dataset)} prompts across {len(batches)} batch(es).")


In [ ]:
class DiffusionModelWrapper(nn.Module):
    def __init__(self, model_id: str, role: str) -> None:
        super().__init__()
        self.role = role
        self.pipeline = StableDiffusion3Pipeline.from_pretrained(
            model_id,
            torch_dtype=_resolve_dtype(DTYPE_NAME),
            revision=REVISION,
            variant=VARIANT,
            cache_dir=CACHE_DIR,
            local_files_only=LOCAL_FILES_ONLY,
            use_safetensors=USE_SAFETENSORS,
            low_cpu_mem_usage=LOW_CPU_MEM_USAGE,
        )
        self.transformer = self.pipeline.transformer

        if FREEZE_TEXT_ENCODERS:
            for encoder_name in ("text_encoder", "text_encoder_2", "text_encoder_3"):
                encoder = getattr(self.pipeline, encoder_name, None)
                if encoder is None:
                    continue
                for param in encoder.parameters():
                    param.requires_grad = False

        if FREEZE_VAE and getattr(self.pipeline, "vae", None) is not None:
            for param in self.pipeline.vae.parameters():
                param.requires_grad = False

        train_transformer = role == "student"
        for param in self.transformer.parameters():
            param.requires_grad = train_transformer

    def encode_prompts(self, prompts: list[str], device: torch.device) -> dict[str, torch.Tensor]:
        for encoder_name in ("text_encoder", "text_encoder_2", "text_encoder_3"):
            encoder = getattr(self.pipeline, encoder_name, None)
            if encoder is not None:
                encoder.to(device=device, dtype=torch.float32)
        with torch.no_grad(), autocast("cuda", enabled=False):
            prompt_embeds, _, pooled_prompt_embeds, _ = self.pipeline.encode_prompt(
                prompt=prompts,
                prompt_2=prompts,
                prompt_3=prompts,
                device=device,
                num_images_per_prompt=1,
                do_classifier_free_guidance=False,
                max_sequence_length=MAX_SEQUENCE_LENGTH,
            )
        return {
            "prompt_embeds": prompt_embeds.to(dtype=torch.float32),
            "pooled_prompt_embeds": pooled_prompt_embeds.to(dtype=torch.float32),
        }

    def prepare_initial_latents(self, batch_size: int, seed: int, device: torch.device) -> torch.Tensor:
        generator = torch.Generator(device=device)
        generator.manual_seed(seed)
        return self.pipeline.prepare_latents(
            batch_size=batch_size,
            num_channels_latents=self.transformer.config.in_channels,
            height=IMAGE_HEIGHT,
            width=IMAGE_WIDTH,
            dtype=_resolve_dtype(DTYPE_NAME),
            device=device,
            generator=generator,
            latents=None,
        ) * INITIAL_NOISE_SCALE

    def get_timesteps(self, device: torch.device) -> list[Any]:
        self.pipeline.scheduler.set_timesteps(TRAIN_NUM_INFERENCE_STEPS, device=device)
        return list(self.pipeline.scheduler.timesteps)

    def predict(
        self,
        latents: torch.Tensor,
        timestep: Any,
        prompt_embeds: torch.Tensor,
        pooled_prompt_embeds: torch.Tensor,
    ) -> torch.Tensor:
        model_dtype = next(self.transformer.parameters()).dtype
        latent_input = latents.to(dtype=model_dtype)
        prompt_embeds = prompt_embeds.to(device=latents.device, dtype=model_dtype)
        pooled_prompt_embeds = pooled_prompt_embeds.to(device=latents.device, dtype=model_dtype)
        if isinstance(timestep, torch.Tensor):
            timestep_tensor = timestep.to(device=latents.device, dtype=model_dtype)
        else:
            timestep_tensor = torch.tensor(timestep, device=latents.device, dtype=model_dtype)
        timestep_tensor = timestep_tensor.expand(latent_input.shape[0])

        epsilon = self.transformer(
            hidden_states=latent_input,
            timestep=timestep_tensor,
            encoder_hidden_states=prompt_embeds,
            pooled_projections=pooled_prompt_embeds,
            return_dict=False,
        )[0].to(dtype=latents.dtype)
        return epsilon

    def step_latents(self, latents: torch.Tensor, epsilon: torch.Tensor, timestep: Any) -> torch.Tensor:
        return self.pipeline.scheduler.step(
            model_output=epsilon.to(latents.dtype),
            timestep=timestep,
            sample=latents,
            return_dict=False,
        )[0]


def save_checkpoint(path: Path, student: DiffusionModelWrapper, optimizer: torch.optim.Optimizer, step: int) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "student_transformer": student.transformer.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step": step,
        "run_name": RUN_NAME,
    }, path)


def save_eval_images(
    *,
    model: DiffusionModelWrapper,
    model_name: str,
    prompts: list[str],
    output_dir: Path,
    seed_base: int,
    device: torch.device,
) -> None:
    model.eval()
    model.pipeline.to(device)
    model_dir = output_dir / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    with torch.inference_mode():
        for prompt_idx, prompt in enumerate(prompts):
            generator = torch.Generator(device=device).manual_seed(seed_base + prompt_idx)
            result = model.pipeline(
                prompt=prompt,
                prompt_2=prompt,
                prompt_3=prompt,
                num_inference_steps=EVAL_NUM_INFERENCE_STEPS,
                height=IMAGE_HEIGHT,
                width=IMAGE_WIDTH,
                generator=generator,
            )
            result.images[0].save(model_dir / f"{prompt_idx:02d}_{_slugify(prompt)}.png")

    with (model_dir / "manifest.json").open("w", encoding="utf-8") as handle:
        json.dump({
            "model": model_name,
            "seed_base": seed_base,
            "num_inference_steps": EVAL_NUM_INFERENCE_STEPS,
            "height": IMAGE_HEIGHT,
            "width": IMAGE_WIDTH,
            "prompts": prompts,
        }, handle, indent=2, sort_keys=True)


In [ ]:
set_seed(SEED)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

teacher = DiffusionModelWrapper(TEACHER_MODEL_ID, role="teacher").to(DEVICE)
student = DiffusionModelWrapper(STUDENT_MODEL_ID, role="student").to(DEVICE)
teacher.pipeline.to(DEVICE)
student.pipeline.to(DEVICE)
teacher.eval()
student.train()

trainable_params = [param for param in student.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

model_dtype = _resolve_dtype(DTYPE_NAME)
use_autocast = MIXED_PRECISION and model_dtype in (torch.float16, torch.bfloat16)
use_grad_scaler = MIXED_PRECISION and model_dtype == torch.float16
scaler = GradScaler("cuda", enabled=use_grad_scaler)

global_step = 0
optimizer.zero_grad(set_to_none=True)

while global_step < MAX_TRAIN_STEPS:
    batch = batches[global_step % len(batches)]
    prompts = [row.prompt for row in batch]
    step_seed = SEED + global_step

    with autocast("cuda", enabled=use_autocast):
        latents = student.prepare_initial_latents(
            batch_size=len(prompts),
            seed=step_seed,
            device=DEVICE,
        )
        embeddings = student.encode_prompts(prompts, device=DEVICE)
        timesteps = student.get_timesteps(device=DEVICE)

        eps_terms = []

        for timestep in timesteps:
            with torch.no_grad():
                teacher_eps = teacher.predict(
                    latents=latents,
                    timestep=timestep,
                    prompt_embeds=embeddings["prompt_embeds"],
                    pooled_prompt_embeds=embeddings["pooled_prompt_embeds"],
                )

            student_eps = student.predict(
                latents=latents,
                timestep=timestep,
                prompt_embeds=embeddings["prompt_embeds"],
                pooled_prompt_embeds=embeddings["pooled_prompt_embeds"],
            )

            eps_terms.append(F.mse_loss(student_eps, teacher_eps))
            latents = student.step_latents(latents=latents, epsilon=student_eps, timestep=timestep)

        epsilon_mse_loss = torch.stack(eps_terms).mean()
        total_loss = epsilon_mse_loss
        loss = total_loss / GRAD_ACCUM_STEPS

    scaler.scale(loss).backward()

    grad_norm = None
    if (global_step + 1) % GRAD_ACCUM_STEPS == 0:
        if use_grad_scaler:
            scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
        if use_grad_scaler:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    metrics = {
        "step": float(global_step),
        "total_loss": float(total_loss.detach().cpu().item()),
        "epsilon_mse_loss": float(epsilon_mse_loss.detach().cpu().item()),
        "lr": float(optimizer.param_groups[0]["lr"]),
        "grad_norm": None if grad_norm is None else float(grad_norm.detach().cpu().item()),
    }

    if global_step % LOG_EVERY == 0:
        print(json.dumps(metrics, sort_keys=True))
        append_metric(METRICS_PATH, metrics)

    if global_step > 0 and global_step % SAVE_EVERY == 0:
        save_checkpoint(CHECKPOINT_DIR / f"step_{global_step:07d}.pt", student, optimizer, global_step)

    global_step += 1

final_ckpt = CHECKPOINT_DIR / "final.pt"
save_checkpoint(final_ckpt, student, optimizer, global_step)

save_eval_images(
    model=teacher,
    model_name="teacher",
    prompts=EVAL_PROMPTS,
    output_dir=EVAL_OUTPUT_DIR,
    seed_base=SEED + 10_000,
    device=DEVICE,
)

del teacher
del student
clear_cuda()

original_student = DiffusionModelWrapper(STUDENT_MODEL_ID, role="student").to(DEVICE)
save_eval_images(
    model=original_student,
    model_name="original_student",
    prompts=EVAL_PROMPTS,
    output_dir=EVAL_OUTPUT_DIR,
    seed_base=SEED + 10_000,
    device=DEVICE,
)

del original_student
clear_cuda()

distilled_student = DiffusionModelWrapper(STUDENT_MODEL_ID, role="student").to(DEVICE)
distilled_state = torch.load(final_ckpt, map_location="cpu")
distilled_student.transformer.load_state_dict(distilled_state["student_transformer"])
save_eval_images(
    model=distilled_student,
    model_name="distilled_student",
    prompts=EVAL_PROMPTS,
    output_dir=EVAL_OUTPUT_DIR,
    seed_base=SEED + 10_000,
    device=DEVICE,
)

del distilled_student
clear_cuda()

summary = {
    "experiment_name": EXPERIMENT_NAME,
    "run_name": RUN_NAME,
    "config_hash": _config_hash(),
    "num_prompts": len(dataset),
    "max_train_steps": MAX_TRAIN_STEPS,
    "checkpoint": str(final_ckpt),
    "metrics_jsonl": str(METRICS_PATH),
    "eval_output_dir": str(EVAL_OUTPUT_DIR),
    "eval_prompts": EVAL_PROMPTS,
}

with SUMMARY_PATH.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2, sort_keys=True)

archive_path = shutil.make_archive(str(ZIP_BASENAME), "zip", root_dir=EVAL_OUTPUT_DIR.parent, base_dir=EVAL_OUTPUT_DIR.name)
print(json.dumps(summary, indent=2, sort_keys=True))
print(f"Zipped evaluation images to {archive_path}")

try:
    from google.colab import files
    files.download(archive_path)
except Exception as exc:
    print(f"Skipping automatic download: {exc}")
